In [69]:
# ============================================================
# Cell 1 - Library dan konfigurasi utama
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output


# ============================================================
# ROOT PATH
# ============================================================

DBSCAN_INPUT_ROOT = Path("/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset")
SAMPLED_INPUT_ROOT = Path("/media/spell/Spell-lab/Lidar/E.Sampled Dataset")


# ============================================================
# PILIH N TARGET
# ============================================================

N_TARGET = 1024
N_FOLDER_NAME = f"N{N_TARGET:04d}"


# ============================================================
# PILIH FILE UNTUK DIVISUALISASIKAN
# ============================================================
# DATASET_MODE: "development" atau "testing"

DATASET_MODE = "development"

ACTIVITY = "Duduk"
SUBJECT = "Teguh"
FILE_ID = 5

# Khusus testing
ROOM = "Controlled Room"


# ============================================================
# VISUAL CONFIG
# ============================================================

COORD_COLUMNS = ["X_corr", "Y_corr", "Z_level"]
COLOR_COLUMN = "Z_level"

ROI_X_MIN, ROI_X_MAX = 0.0, 3.0
ROI_Y_MIN, ROI_Y_MAX = -1.5, 1.5
ROI_Z_MIN, ROI_Z_MAX = 0.0, 2.0

MARKER_SIZE_BEFORE = 2
MARKER_SIZE_AFTER = 2

MAX_POINTS_PLOT = 20000

print("===== BEFORE-AFTER SAMPLING VISUALIZATION CONFIG =====")
print(f"N_TARGET     : {N_TARGET}")
print(f"N folder     : {N_FOLDER_NAME}")
print(f"Dataset mode : {DATASET_MODE}")
print(f"Activity     : {ACTIVITY}")
print(f"Subject      : {SUBJECT}")
print(f"File ID      : {FILE_ID}")
if DATASET_MODE == "testing":
    print(f"Room         : {ROOM}")

===== BEFORE-AFTER SAMPLING VISUALIZATION CONFIG =====
N_TARGET     : 1024
N folder     : N1024
Dataset mode : development
Activity     : Duduk
Subject      : Teguh
File ID      : 5


In [70]:
# ============================================================
# Cell 2 - Build path before, after, dan summary
# ============================================================

file_name = f"{FILE_ID}.csv"

if DATASET_MODE == "development":
    before_path = (
        DBSCAN_INPUT_ROOT
        / "Dataset Development"
        / ACTIVITY
        / SUBJECT
        / file_name
    )

    after_path = (
        SAMPLED_INPUT_ROOT
        / N_FOLDER_NAME
        / "Dataset Development"
        / ACTIVITY
        / SUBJECT
        / file_name
    )

    summary_path = (
        SAMPLED_INPUT_ROOT
        / N_FOLDER_NAME
        / "Summary"
        / "per_file_frame_summary"
        / "Dataset Development"
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}_sampling_summary.csv"
    )

elif DATASET_MODE == "testing":
    before_path = (
        DBSCAN_INPUT_ROOT
        / "Dataset Testing"
        / ROOM
        / ACTIVITY
        / SUBJECT
        / file_name
    )

    after_path = (
        SAMPLED_INPUT_ROOT
        / N_FOLDER_NAME
        / "Dataset Testing"
        / ROOM
        / ACTIVITY
        / SUBJECT
        / file_name
    )

    summary_path = (
        SAMPLED_INPUT_ROOT
        / N_FOLDER_NAME
        / "Summary"
        / "per_file_frame_summary"
        / "Dataset Testing"
        / ROOM
        / ACTIVITY
        / SUBJECT
        / f"{FILE_ID}_sampling_summary.csv"
    )

else:
    raise ValueError("DATASET_MODE harus 'development' atau 'testing'.")


print("===== PATH CHECK =====")
print("Before DBSCAN-main-cluster:")
print(before_path)
print("Exists:", before_path.exists())

print("\nAfter fixed-N sampling:")
print(after_path)
print("Exists:", after_path.exists())

print("\nSampling summary:")
print(summary_path)
print("Exists:", summary_path.exists())

if not before_path.exists():
    raise FileNotFoundError(f"Before file tidak ditemukan: {before_path}")

if not after_path.exists():
    raise FileNotFoundError(f"After file tidak ditemukan: {after_path}")

if not summary_path.exists():
    raise FileNotFoundError(f"Summary file tidak ditemukan: {summary_path}")

===== PATH CHECK =====
Before DBSCAN-main-cluster:
/media/spell/Spell-lab/Lidar/D.DBSCAN Dataset/Dataset Development/Duduk/Teguh/5.csv
Exists: True

After fixed-N sampling:
/media/spell/Spell-lab/Lidar/E.Sampled Dataset/N1024/Dataset Development/Duduk/Teguh/5.csv
Exists: True

Sampling summary:
/media/spell/Spell-lab/Lidar/E.Sampled Dataset/N1024/Summary/per_file_frame_summary/Dataset Development/Duduk/Teguh/5_sampling_summary.csv
Exists: True


In [71]:
# ============================================================
# Cell 3 - Load data dan validasi kolom
# ============================================================

REQUIRED_COLUMNS = [
    "frame_id",
    "Timestamp",
    "X",
    "Y",
    "Z",
    "X_corr",
    "Y_corr",
    "Z_corr",
    "Z_level",
    "Reflectivity",
]

before_df = pd.read_csv(before_path)
after_df = pd.read_csv(after_path)
summary_df = pd.read_csv(summary_path)

for name, df in [("before_df", before_df), ("after_df", after_df)]:
    missing_cols = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f"{name} missing columns: {missing_cols}")

before_df = before_df[REQUIRED_COLUMNS].copy()
after_df = after_df[REQUIRED_COLUMNS].copy()

for df in [before_df, after_df]:
    for col in ["frame_id", "Timestamp", "X_corr", "Y_corr", "Z_corr", "Z_level", "Reflectivity"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df.dropna(subset=["frame_id", "X_corr", "Y_corr", "Z_level"], inplace=True)
    df["frame_id"] = df["frame_id"].astype(int)

before_frame_ids = set(before_df["frame_id"].unique().tolist())
after_frame_ids = set(after_df["frame_id"].unique().tolist())

frame_ids = sorted(list(before_frame_ids.intersection(after_frame_ids)))

if len(frame_ids) == 0:
    raise ValueError("Tidak ada frame_id yang overlap antara before dan after.")

print("===== DATA LOADED =====")
print(f"Before shape        : {before_df.shape}")
print(f"After shape         : {after_df.shape}")
print(f"Summary shape       : {summary_df.shape}")
print(f"Overlap frame count : {len(frame_ids)}")
print(f"First frame         : {frame_ids[0]}")
print(f"Last frame          : {frame_ids[-1]}")

display(before_df.head())
display(after_df.head())
display(summary_df.head())

===== DATA LOADED =====
Before shape        : (34723, 10)
After shape         : (61440, 10)
Summary shape       : (60, 15)
Overlap frame count : 60
First frame         : 0
Last frame          : 59


,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,172798514380,0.963,-0.096,1.171,1.367660,-0.096,0.654305,1.545384,24.0
1,0,172798514380,1.099,-0.240,1.033,1.432597,-0.240,0.471758,1.362837,9.0
2,0,172798514380,1.048,-0.149,1.279,1.490339,-0.149,0.716264,1.607342,3.0
3,0,172798514380,1.106,-0.215,1.035,1.439786,-0.215,0.470613,1.361691,62.0
4,0,172798514380,0.981,-0.119,1.196,1.394539,-0.119,0.669356,1.560434,6.0


,frame_id,Timestamp,X,Y,Z,X_corr,Y_corr,Z_corr,Z_level,Reflectivity
0,0,172798514380,0.963,-0.096,1.171,1.367660,-0.096,0.654305,1.545384,24.0
1,0,172798514380,1.099,-0.240,1.033,1.432597,-0.240,0.471758,1.362837,9.0
2,0,172798514380,1.048,-0.149,1.279,1.490339,-0.149,0.716264,1.607342,3.0
3,0,172798514380,1.106,-0.215,1.035,1.439786,-0.215,0.470613,1.361691,62.0
4,0,172798514380,0.981,-0.119,1.196,1.394539,-0.119,0.669356,1.560434,6.0


,dataset,room,activity,subject,file_id,file_name,frame_id,n_original_points,n_target,sampling_status,sampling_mode,n_sampled_points,n_added,n_removed,duplicate_ratio
0,development,NaN,Duduk,Teguh,5,5.csv,0,832,1024,valid,upsample_bootstrap_retain_all,1024,192,0,0.187500
1,development,NaN,Duduk,Teguh,5,5.csv,1,828,1024,valid,upsample_bootstrap_retain_all,1024,196,0,0.191406
2,development,NaN,Duduk,Teguh,5,5.csv,2,822,1024,valid,upsample_bootstrap_retain_all,1024,202,0,0.197266
3,development,NaN,Duduk,Teguh,5,5.csv,3,834,1024,valid,upsample_bootstrap_retain_all,1024,190,0,0.185547
4,development,NaN,Duduk,Teguh,5,5.csv,4,829,1024,valid,upsample_bootstrap_retain_all,1024,195,0,0.190430


In [72]:
# ============================================================
# Cell 4 - Build frame-level comparison summary
# ============================================================

before_count_df = (
    before_df
    .groupby("frame_id")
    .size()
    .reset_index(name="before_points")
)

after_count_df = (
    after_df
    .groupby("frame_id")
    .size()
    .reset_index(name="after_points")
)

comparison_df = before_count_df.merge(
    after_count_df,
    on="frame_id",
    how="inner"
)

summary_needed_cols = [
    "frame_id",
    "sampling_mode",
    "n_original_points",
    "n_target",
    "n_sampled_points",
    "n_added",
    "n_removed",
    "duplicate_ratio",
]

available_summary_cols = [col for col in summary_needed_cols if col in summary_df.columns]

comparison_df = comparison_df.merge(
    summary_df[available_summary_cols],
    on="frame_id",
    how="left"
)

comparison_df = comparison_df.sort_values("frame_id").reset_index(drop=True)

print("===== COMPARISON SUMMARY =====")
display(comparison_df.head())
display(comparison_df["sampling_mode"].value_counts(dropna=False).to_frame("count"))

===== COMPARISON SUMMARY =====


,frame_id,before_points,after_points,sampling_mode,n_original_points,n_target,n_sampled_points,n_added,n_removed,duplicate_ratio
0,0,832,1024,upsample_bootstrap_retain_all,832,1024,1024,192,0,0.187500
1,1,828,1024,upsample_bootstrap_retain_all,828,1024,1024,196,0,0.191406
2,2,822,1024,upsample_bootstrap_retain_all,822,1024,1024,202,0,0.197266
3,3,834,1024,upsample_bootstrap_retain_all,834,1024,1024,190,0,0.185547
4,4,829,1024,upsample_bootstrap_retain_all,829,1024,1024,195,0,0.190430


,count
sampling_mode,
upsample_bootstrap_retain_all,60


In [73]:
# ============================================================
# Cell 5 - Fungsi summary HTML per frame
# ============================================================

def format_value(value):
    if pd.isna(value):
        return "-"
    if isinstance(value, float):
        return f"{value:.6f}"
    return str(value)


def make_summary_html(frame_id):
    row = comparison_df[comparison_df["frame_id"] == frame_id].iloc[0]

    rows = [
        ("frame_id", frame_id),
        ("before_points", row.get("before_points", np.nan)),
        ("after_points", row.get("after_points", np.nan)),
        ("N target", row.get("n_target", N_TARGET)),
        ("sampling_mode", row.get("sampling_mode", "-")),
        ("n_original_points", row.get("n_original_points", np.nan)),
        ("n_sampled_points", row.get("n_sampled_points", np.nan)),
        ("n_added", row.get("n_added", np.nan)),
        ("n_removed", row.get("n_removed", np.nan)),
        ("duplicate_ratio", row.get("duplicate_ratio", np.nan)),
    ]

    html = """
    <div style="font-family: Arial; font-size: 13px;">
    <h3>Before vs After Sampling Summary</h3>
    <table style="border-collapse: collapse;">
    """

    for key, value in rows:
        html += f"""
        <tr>
            <td style="border: 1px solid #ccc; padding: 4px 8px;"><b>{key}</b></td>
            <td style="border: 1px solid #ccc; padding: 4px 8px;">{format_value(value)}</td>
        </tr>
        """

    html += """
    </table>
    </div>
    """

    return HTML(html)

In [74]:
# ============================================================
# Cell 6 - Fungsi plot 3D side-by-side
# ============================================================

def sample_for_plot(df, max_points=MAX_POINTS_PLOT):
    if len(df) > max_points:
        return df.sample(max_points, random_state=42).copy()
    return df.copy()


def make_before_after_figure(frame_id):
    before_frame = before_df[before_df["frame_id"] == frame_id].copy()
    after_frame = after_df[after_df["frame_id"] == frame_id].copy()

    before_frame = sample_for_plot(before_frame)
    after_frame = sample_for_plot(after_frame)

    fig = make_subplots(
        rows=1,
        cols=2,
        specs=[[{"type": "scene"}, {"type": "scene"}]],
        subplot_titles=(
            f"Before Sampling | DBSCAN Main Cluster | M={len(before_frame)}",
            f"After Sampling | Fixed-N={N_TARGET} | N={len(after_frame)}",
        ),
        horizontal_spacing=0.02,
    )

    fig.add_trace(
        go.Scatter3d(
            x=before_frame["X_corr"],
            y=before_frame["Y_corr"],
            z=before_frame["Z_level"],
            mode="markers",
            marker=dict(
                size=MARKER_SIZE_BEFORE,
                color=before_frame[COLOR_COLUMN],
                colorscale="Turbo",
                opacity=0.85,
                colorbar=dict(title="Z_level", x=0.46),
            ),
            text=[
                f"Before<br>"
                f"frame_id={fid}<br>"
                f"X_corr={x:.3f}<br>"
                f"Y_corr={y:.3f}<br>"
                f"Z_level={z:.3f}<br>"
                f"Reflectivity={r}"
                for fid, x, y, z, r in zip(
                    before_frame["frame_id"],
                    before_frame["X_corr"],
                    before_frame["Y_corr"],
                    before_frame["Z_level"],
                    before_frame["Reflectivity"],
                )
            ],
            hoverinfo="text",
            name="Before",
        ),
        row=1,
        col=1,
    )

    fig.add_trace(
        go.Scatter3d(
            x=after_frame["X_corr"],
            y=after_frame["Y_corr"],
            z=after_frame["Z_level"],
            mode="markers",
            marker=dict(
                size=MARKER_SIZE_AFTER,
                color=after_frame[COLOR_COLUMN],
                colorscale="Turbo",
                opacity=0.85,
                colorbar=dict(title="Z_level", x=1.02),
            ),
            text=[
                f"After<br>"
                f"frame_id={fid}<br>"
                f"X_corr={x:.3f}<br>"
                f"Y_corr={y:.3f}<br>"
                f"Z_level={z:.3f}<br>"
                f"Reflectivity={r}"
                for fid, x, y, z, r in zip(
                    after_frame["frame_id"],
                    after_frame["X_corr"],
                    after_frame["Y_corr"],
                    after_frame["Z_level"],
                    after_frame["Reflectivity"],
                )
            ],
            hoverinfo="text",
            name="After",
        ),
        row=1,
        col=2,
    )

    scene_common = dict(
        xaxis=dict(title="X_corr", range=[ROI_X_MIN, ROI_X_MAX]),
        yaxis=dict(title="Y_corr", range=[ROI_Y_MIN, ROI_Y_MAX]),
        zaxis=dict(title="Z_level", range=[ROI_Z_MIN, ROI_Z_MAX]),
        aspectmode="manual",
        aspectratio=dict(x=1.5, y=1.5, z=1.0),
    )

    fig.update_layout(
        title=(
            f"Before vs After Fixed-N Sampling<br>"
            f"Frame ID: {frame_id} | N_TARGET={N_TARGET}"
        ),
        scene=scene_common,
        scene2=scene_common,
        width=1300,
        height=750,
        margin=dict(l=0, r=0, b=0, t=90),
        showlegend=False,
    )

    return fig

In [75]:
# ============================================================
# Cell 7 - GUI Play, Pause, Prev, Next, Slider
# ============================================================

frame_index_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    description="Frame idx:",
    continuous_update=False,
    layout=widgets.Layout(width="650px"),
)

play_widget = widgets.Play(
    value=0,
    min=0,
    max=len(frame_ids) - 1,
    step=1,
    interval=600,
    description="Play",
    disabled=False,
)

widgets.jslink((play_widget, "value"), (frame_index_slider, "value"))

prev_button = widgets.Button(
    description="Prev Frame",
    tooltip="Go to previous frame",
    icon="arrow-left",
)

next_button = widgets.Button(
    description="Next Frame",
    tooltip="Go to next frame",
    icon="arrow-right",
)

pause_button = widgets.Button(
    description="Pause",
    tooltip="Pause animation",
    icon="pause",
)

frame_label = widgets.HTML()

plot_output = widgets.Output()
summary_output = widgets.Output()


def update_frame_label():
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]

    frame_label.value = (
        f"<b>Current:</b> index {idx + 1}/{len(frame_ids)} | "
        f"<b>frame_id:</b> {frame_id} | "
        f"<b>N_TARGET:</b> {N_TARGET}"
    )


def render_current_frame(change=None):
    idx = frame_index_slider.value
    frame_id = frame_ids[idx]

    update_frame_label()

    with summary_output:
        clear_output(wait=True)
        display(make_summary_html(frame_id))

    with plot_output:
        clear_output(wait=True)
        fig = make_before_after_figure(frame_id)
        fig.show()


def on_prev_clicked(button):
    current = frame_index_slider.value
    if current > frame_index_slider.min:
        frame_index_slider.value = current - 1


def on_next_clicked(button):
    current = frame_index_slider.value
    if current < frame_index_slider.max:
        frame_index_slider.value = current + 1


def on_pause_clicked(button):
    play_widget.value = frame_index_slider.value
    play_widget.disabled = True
    play_widget.disabled = False


prev_button.on_click(on_prev_clicked)
next_button.on_click(on_next_clicked)
pause_button.on_click(on_pause_clicked)

frame_index_slider.observe(render_current_frame, names="value")

controls = widgets.VBox([
    widgets.HBox([play_widget, pause_button, frame_index_slider]),
    widgets.HBox([prev_button, next_button]),
    frame_label,
])

display(controls)
display(summary_output)
display(plot_output)

render_current_frame()

Output()

Output()

In [76]:
# ============================================================
# Cell 8 - Cari frame ekstrem untuk validasi
# ============================================================

print("===== MOST UPSAMPLED FRAMES =====")
if "n_added" in comparison_df.columns:
    display(
        comparison_df
        .sort_values("n_added", ascending=False)
        .head(15)
    )

print("===== MOST DOWNSAMPLED FRAMES =====")
if "n_removed" in comparison_df.columns:
    display(
        comparison_df
        .sort_values("n_removed", ascending=False)
        .head(15)
    )

print("===== LOWEST BEFORE POINTS =====")
display(
    comparison_df
    .sort_values("before_points", ascending=True)
    .head(15)
)

print("===== HIGHEST BEFORE POINTS =====")
display(
    comparison_df
    .sort_values("before_points", ascending=False)
    .head(15)
)

===== MOST UPSAMPLED FRAMES =====


,frame_id,before_points,after_points,sampling_mode,n_original_points,n_target,n_sampled_points,n_added,n_removed,duplicate_ratio
31,31,428,1024,upsample_bootstrap_retain_all,428,1024,1024,596,0,0.582031
53,53,436,1024,upsample_bootstrap_retain_all,436,1024,1024,588,0,0.574219
54,54,439,1024,upsample_bootstrap_retain_all,439,1024,1024,585,0,0.571289
48,48,440,1024,upsample_bootstrap_retain_all,440,1024,1024,584,0,0.570312
56,56,441,1024,upsample_bootstrap_retain_all,441,1024,1024,583,0,0.569336
29,29,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
58,58,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
28,28,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
55,55,443,1024,upsample_bootstrap_retain_all,443,1024,1024,581,0,0.567383
57,57,446,1024,upsample_bootstrap_retain_all,446,1024,1024,578,0,0.564453


===== MOST DOWNSAMPLED FRAMES =====


,frame_id,before_points,after_points,sampling_mode,n_original_points,n_target,n_sampled_points,n_added,n_removed,duplicate_ratio
0,0,832,1024,upsample_bootstrap_retain_all,832,1024,1024,192,0,0.187500
1,1,828,1024,upsample_bootstrap_retain_all,828,1024,1024,196,0,0.191406
32,32,450,1024,upsample_bootstrap_retain_all,450,1024,1024,574,0,0.560547
33,33,446,1024,upsample_bootstrap_retain_all,446,1024,1024,578,0,0.564453
34,34,480,1024,upsample_bootstrap_retain_all,480,1024,1024,544,0,0.531250
35,35,462,1024,upsample_bootstrap_retain_all,462,1024,1024,562,0,0.548828
36,36,485,1024,upsample_bootstrap_retain_all,485,1024,1024,539,0,0.526367
37,37,470,1024,upsample_bootstrap_retain_all,470,1024,1024,554,0,0.541016
38,38,469,1024,upsample_bootstrap_retain_all,469,1024,1024,555,0,0.541992
39,39,468,1024,upsample_bootstrap_retain_all,468,1024,1024,556,0,0.542969


===== LOWEST BEFORE POINTS =====


,frame_id,before_points,after_points,sampling_mode,n_original_points,n_target,n_sampled_points,n_added,n_removed,duplicate_ratio
31,31,428,1024,upsample_bootstrap_retain_all,428,1024,1024,596,0,0.582031
53,53,436,1024,upsample_bootstrap_retain_all,436,1024,1024,588,0,0.574219
54,54,439,1024,upsample_bootstrap_retain_all,439,1024,1024,585,0,0.571289
48,48,440,1024,upsample_bootstrap_retain_all,440,1024,1024,584,0,0.570312
56,56,441,1024,upsample_bootstrap_retain_all,441,1024,1024,583,0,0.569336
29,29,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
58,58,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
28,28,442,1024,upsample_bootstrap_retain_all,442,1024,1024,582,0,0.568359
55,55,443,1024,upsample_bootstrap_retain_all,443,1024,1024,581,0,0.567383
57,57,446,1024,upsample_bootstrap_retain_all,446,1024,1024,578,0,0.564453


===== HIGHEST BEFORE POINTS =====


,frame_id,before_points,after_points,sampling_mode,n_original_points,n_target,n_sampled_points,n_added,n_removed,duplicate_ratio
6,6,870,1024,upsample_bootstrap_retain_all,870,1024,1024,154,0,0.150391
7,7,867,1024,upsample_bootstrap_retain_all,867,1024,1024,157,0,0.153320
12,12,836,1024,upsample_bootstrap_retain_all,836,1024,1024,188,0,0.183594
13,13,835,1024,upsample_bootstrap_retain_all,835,1024,1024,189,0,0.184570
10,10,834,1024,upsample_bootstrap_retain_all,834,1024,1024,190,0,0.185547
3,3,834,1024,upsample_bootstrap_retain_all,834,1024,1024,190,0,0.185547
5,5,833,1024,upsample_bootstrap_retain_all,833,1024,1024,191,0,0.186523
8,8,833,1024,upsample_bootstrap_retain_all,833,1024,1024,191,0,0.186523
0,0,832,1024,upsample_bootstrap_retain_all,832,1024,1024,192,0,0.187500
11,11,831,1024,upsample_bootstrap_retain_all,831,1024,1024,193,0,0.188477


In [77]:
# ============================================================
# Cell 9 - Jump ke frame tertentu
# ============================================================

def jump_to_frame_id(target_frame_id):
    if target_frame_id not in frame_ids:
        print(f"Frame ID {target_frame_id} tidak ditemukan.")
        print(f"Available range: {frame_ids[0]} sampai {frame_ids[-1]}")
        return

    idx = frame_ids.index(target_frame_id)
    frame_index_slider.value = idx
    print(f"Jumped to frame_id={target_frame_id}, index={idx}")


# Contoh:
# jump_to_frame_id(31)

In [78]:
# ============================================================
# Cell 10 - Ringkasan sampling mode untuk file ini
# ============================================================

print("===== SAMPLING MODE SUMMARY FOR THIS FILE =====")
display(comparison_df["sampling_mode"].value_counts(dropna=False).to_frame("count"))

print("===== POINT COUNT SUMMARY =====")
display(
    comparison_df[
        [
            "before_points",
            "after_points",
            "n_added",
            "n_removed",
            "duplicate_ratio",
        ]
    ].describe()
)

===== SAMPLING MODE SUMMARY FOR THIS FILE =====


,count
sampling_mode,
upsample_bootstrap_retain_all,60


===== POINT COUNT SUMMARY =====


,before_points,after_points,n_added,n_removed,duplicate_ratio
count,60.000000,60.0,60.000000,60.0,60.000000
mean,578.716667,1024.0,445.283333,0.0,0.434847
std,165.634895,0.0,165.634895,0.0,0.161753
min,428.000000,1024.0,154.000000,0.0,0.150391
25%,454.000000,1024.0,223.750000,0.0,0.218506
50%,471.500000,1024.0,552.500000,0.0,0.539551
75%,800.250000,1024.0,570.000000,0.0,0.556641
max,870.000000,1024.0,596.000000,0.0,0.582031
